# Phishing Detector ML - Cloud Compute Training (Google Colab GPU)

This notebook trains the **5-Layer ReLU MLP Neural Network** and **Character-Level LSTM** on Cloud Compute (Google Colab / Vertex AI) with GPU Acceleration (`torch.cuda`).

### Instructions for Google Colab:
1. In Google Colab menu, click **Runtime** -> **Change runtime type**.
2. Under **Hardware accelerator**, select **GPU (T4 GPU)** and click **Save**.
3. Run all cells below (`Cmd/Ctrl + F9`).

In [ ]:
# Step 1: Install Kagglehub & Verify GPU Availability
!pip install -q kagglehub torch pandas scikit-learn matplotlib

import time
import torch

# Check Device (GPU vs CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU (Running on CPU)'

print('=' * 60)
print(f'Active Device      : {device}')
print(f'GPU Hardware Name  : {gpu_name}')
print(f'PyTorch CUDA Status: {torch.cuda.is_available()}')
print('=' * 60)

In [ ]:
# Step 2: Download Dataset via kagglehub
import kagglehub
import os
import pandas as pd

path = kagglehub.dataset_download('shashwatwork/phishing-dataset-for-machine-learning')
csv_path = os.path.join(path, 'Phishing_Legitimate_full.csv')

df = pd.read_csv(csv_path)
print(f'Dataset successfully loaded! Shape: {df.shape}')
print('Class Distribution:\n', df['CLASS_LABEL'].value_counts())

In [ ]:
# Step 3: Define & Train 5-Layer ReLU MLP on Cloud GPU
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Preprocess Data
X = df.drop(columns=['id', 'CLASS_LABEL'])
y = df['CLASS_LABEL']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_train)
X_te_scaled = scaler.transform(X_test)

# Move Tensors to GPU Device
X_tr_t = torch.tensor(X_tr_scaled, dtype=torch.float32).to(device)
y_tr_t = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1).to(device)
X_te_t = torch.tensor(X_te_scaled, dtype=torch.float32).to(device)
y_te_t = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1).to(device)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=128, shuffle=True)

# Define 5-Layer ReLU Architecture
class PhishingReLU_MLP(nn.Module):
    def __init__(self, input_dim=48):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, 16), nn.BatchNorm1d(16), nn.ReLU(),
            nn.Linear(16, 8), nn.BatchNorm1d(8), nn.ReLU(),
            nn.Linear(8, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model = PhishingReLU_MLP().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.002, weight_decay=1e-4)

print('=' * 60)
print('STARTING MLP CLOUD TRAINING...')
start_time = time.time()

epochs = 50
for epoch in range(1, epochs + 1):
    model.train()
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
        
    if epoch % 10 == 0 or epoch == epochs:
        model.eval()
        with torch.no_grad():
            va_loss = criterion(model(X_te_t), y_te_t).item()
            va_preds = (model(X_te_t).cpu().numpy() >= 0.5).astype(int)
            va_acc = accuracy_score(y_test, va_preds) * 100
            print(f'Epoch {epoch:2d}/{epochs} | Val Loss: {va_loss:.4f} | Val Acc: {va_acc:.2f}%')

total_runtime = time.time() - start_time
print('=' * 60)
print(f'MLP Training Completed!')
print(f'Device Used    : {gpu_name}')
print(f'Total Runtime  : {total_runtime:.2f} seconds')
print('=' * 60)

In [ ]:
# Step 4: Final Evaluation Report
model.eval()
with torch.no_grad():
    final_probs = model(X_te_t).cpu().numpy()
    final_preds = (final_probs >= 0.5).astype(int)

target_names = ['Class 0: Phishing', 'Class 1: Legitimate']
acc = accuracy_score(y_test, final_preds)
auc = roc_auc_score(y_test, final_probs)

print('=' * 60)
print('      CLOUD COMPUTE EVALUATION SUMMARY')
print('=' * 60)
print(f'Hardware Accelerators : {gpu_name}')
print(f'Total Model Runtime   : {total_runtime:.2f}s')
print(f'Accuracy              : {acc * 100:.2f}%')
print(f'ROC-AUC Score         : {auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, final_preds, target_names=target_names))
print('=' * 60)